# API Financeira - BACEN
Este notebook demonstra a construção de uma API em Python que consome dados públicos do Banco Central (BACEN), realiza cálculos financeiros (como juros compostos e variação acumulada) e disponibiliza os resultados via endpoints.

# Cria dicionário códigos BACEN

In [2]:
# Definir o dicionário de indicadores macroeconômicos
indicadores = {
    "SELIC": 11,  # Taxa Selic
    "IPCA": 433   # Índice de Preços ao Consumidor Amplo
}

# Função para buscar dados do BACEN

In [3]:
# Função para buscar dados do BACEN 
def get_bacen_currency(serie_id: int, data_inicial=None, data_final=None):
    """
    Coleta séries temporais do Bacen via API SGS e retorna um DataFrame limpo.
    """
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie_id}/dados?formato=json"
    if data_inicial and data_final:
        url += f"&dataInicial={data_inicial}&dataFinal={data_final}"

    headers = {"Accept": "application/json"}  # Evita erros 406
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Erro ao acessar API do Bacen: {response.status_code}")

    data = response.json()
    df = pd.DataFrame(data)
    df['data'] = pd.to_datetime(df['data'], dayfirst=True)
    df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
    df = df.sort_values('data').reset_index(drop=True)
    return df


In [4]:
# Função para validar dados
def validar_dados(df):
    alertas = []
    if df['valor'].isna().sum() > 0:
        alertas.append(f"{df['valor'].isna().sum()} valores ausentes")
    if df.duplicated(subset=['data']).sum() > 0:
        alertas.append(f"{df.duplicated(subset=['data']).sum()} duplicatas")
    # Outliers simples: Z-score
    df_mean = df['valor'].mean()
    df_std = df['valor'].std()
    if df_std != 0:
        z_scores = (df['valor'] - df_mean) / df_std
        outliers = df[abs(z_scores) > 3]
        if not outliers.empty:
            alertas.append(f"{len(outliers)} outliers detectados: {outliers['valor'].tolist()}")
    return alertas

In [5]:
# Função de análise exploratória
def analise_exploratoria(df):
    return {
        "media": df['valor'].mean(),
        "mediana": df['valor'].median(),
        "minimo": df['valor'].min(),
        "maximo": df['valor'].max(),
        "desvio_padrao": df['valor'].std(),
        "variacao_percentual": ((df['valor'].iloc[-1] - df['valor'].iloc[0]) / df['valor'].iloc[0]) * 100
    }


## Criar rota Flask 

In [6]:
# Inicia o Flask
app = Flask(__name__)

### Indicadores

In [7]:
# Rota Flask  para consultar indicadores econômicos
@app.route('/indicadores', methods=['GET'])
def consultar_indicadores():
    import datetime
    data_inicial = request.args.get("inicio", "01/01/2016")
    data_final = request.args.get("fim", datetime.datetime.today().strftime("%d/%m/%Y"))

    resultados = {}

    for sigla, codigo in indicadores.items():
        try:
            if sigla == "SELIC":
                # Busca SELIC em blocos anuais para evitar erro 406
                df_total = pd.DataFrame()
                ano_inicio = int(data_inicial.split("/")[2])
                ano_fim = int(data_final.split("/")[2])
                for ano in range(ano_inicio, ano_fim + 1):
                    inicio_ano = f"01/01/{ano}"
                    fim_ano = f"31/12/{ano}"
                    try:
                        df = get_bacen_currency(codigo, inicio_ano, fim_ano)
                        df_total = pd.concat([df_total, df])
                    except Exception as e:
                        print(f"Ano {ano} deu erro: {e}")
                df_total = df_total.sort_values("data").reset_index(drop=True)
                df = df_total
            else:
                df = get_bacen_currency(codigo, data_inicial, data_final)

            if df.empty:
                resultados[sigla] = {"erro": "Sem dados disponíveis"}
                continue

            # Limpeza, estatísticas e histórico
            alertas = validar_dados(df)
            estatisticas = analise_exploratoria(df)
            ultimo_valor = df['valor'].iloc[-1]

            resultados[sigla] = {
                "ultimo_valor": ultimo_valor,
                "estatisticas": estatisticas,
                "alertas": alertas,
                "historico": df.to_dict(orient="records")  # <<< histórico completo
            }

        except Exception as e:
            resultados[sigla] = {"erro": str(e)}

    return jsonify(resultados)

In [8]:
# roda Flask em thread para não travar o notebook
def run_app():
    app.run(host='0.0.0.0', port=5000, debug=False)

t = Thread(target=run_app)
t.daemon = True
t.start()

print("API de indicadores rodando! Acesse http://127.0.0.1:5000/indicadores")


API de indicadores rodando! Acesse http://127.0.0.1:5000/indicadores
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.15.11:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Feb/2026 02:49:03] "GET /indicadores HTTP/1.1" 200 -


### Validando Flask

In [14]:
# Testando e chamando a rota  Flask de indicador 
response = requests.get("http://127.0.0.1:5000/indicadores")
resultados = response.json()  # <- JSON retornado pela rota
print("Status code:", response.status_code)


Status code: 200
Resposta bruta:
 {"IPCA":{"alertas":["1 outliers detectados: [1.62]"],"estatisticas":{"desvio_padrao":0.38793179088618823,"maximo":1.62,"media":0.41702479338842974,"mediana":0.36,"minimo":-0.68,"variacao_percentual":-74.01574803149606},"historico":[{"data":"Fri, 01 Jan 2016 00:00:00 GMT","valor":1.27},{"data":"Mon, 01 Feb 2016 00:00:00 GMT","valor":0.9},{"data":"Tue, 01 Mar 2016 00:00:00 GMT","valor":0.43},{"data":"Fri, 01 Apr 2016 00:00:00 GMT","valor":0.61},{"data":"Sun, 01 May 2016 00:00:00 GMT","valor":0.78}


### Tranformando o JSON em DataFrame

# Transformar em DataFrame
# Lista para histórico e estatísticas
historico_linhas = []
estatisticas_linhas = []

for indicador, dados in resultados.items():
    if "erro" in dados:
        continue  # ignora indicadores com erro
    
    # Histórico completo
    if "historico" in dados:
        for registro in dados["historico"]:
            historico_linhas.append({
                "indicador": indicador,
                "data": registro["data"],
                "valor": registro["valor"]
            })
    
    # Estatísticas agregadas
    if "estatisticas" in dados:
        estatisticas_linhas.append({
            "indicador": indicador,
            **dados["estatisticas"]
        })

# Cria os DataFrames
df_historico = pd.DataFrame(historico_linhas)
df_historico['data'] = pd.to_datetime(df_historico['data'])
df_historico = df_historico.sort_values(["indicador", "data"]).reset_index(drop=True)

df_estatisticas = pd.DataFrame(estatisticas_linhas)

# Mostra os DataFrames
print("===== Histórico =====")
print(df_historico.head())
print("\n===== Estatísticas =====")
print(df_estatisticas.head())